In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

# Define schemas
inpatient_schema = StructType([
    StructField("BeneID", StringType(), True),
    StructField("ClaimID", StringType(), True),
    StructField("ClaimStartDt", DateType(), True),
    StructField("ClaimEndDt", DateType(), True),
    StructField("Provider", StringType(), True),
    StructField("InscClaimAmtReimbursed", DoubleType(), True),
    StructField("AttendingPhysician", StringType(), True),
    StructField("OperatingPhysician", StringType(), True),
    StructField("OtherPhysician", StringType(), True),
    StructField("AdmissionDt", DateType(), True),
    StructField("DischargeDt", DateType(), True),
    StructField("DiagnosisGroupCode", StringType(), True),
    *[StructField(f"ClmDiagnosisCode_{i}", StringType(), True) for i in range(1, 11)],
    *[StructField(f"ClmProcedureCode_{i}", StringType(), True) for i in range(1, 7)]
])

outpatient_schema = StructType([
    StructField("BeneID", StringType(), True),
    StructField("ClaimID", StringType(), True),
    StructField("ClaimStartDt", DateType(), True),
    StructField("ClaimEndDt", DateType(), True),
    StructField("Provider", StringType(), True),
    StructField("InscClaimAmtReimbursed", DoubleType(), True),
    StructField("AttendingPhysician", StringType(), True),
    StructField("OperatingPhysician", StringType(), True),
    StructField("OtherPhysician", StringType(), True),
    StructField("ClmAdmitDiagnosisCode", StringType(), True),
    StructField("DeductibleAmtPaid", StringType(), True),
    *[StructField(f"ClmDiagnosisCode_{i}", StringType(), True) for i in range(1, 11)],
    *[StructField(f"ClmProcedureCode_{i}", StringType(), True) for i in range(1, 7)]
])

beneficiary_schema = StructType([
    StructField("BeneID", StringType(), False),
    StructField("DOB", DateType(), True),
    StructField("DOD", DateType(), True),
    StructField("Gender", IntegerType(), True),
    StructField("Race", IntegerType(), True),
    StructField("RenalDiseaseIndicator", StringType(), True),
    StructField("State", IntegerType(), True),
    StructField("County", IntegerType(), True),
    StructField("NoOfMonths_PartACov", IntegerType(), True),
    StructField("NoOfMonths_PartBCov", IntegerType(), True),
    *[StructField(f"{condition}", IntegerType(), True) for condition in [
        "ChronicCond_Alzheimer", "ChronicCond_Heartfailure", "ChronicCond_KidneyDisease",
        "ChronicCond_Cancer", "ChronicCond_ObstrPulmonary", "ChronicCond_Depression",
        "ChronicCond_Diabetes", "ChronicCond_IschemicHeart", "ChronicCond_Osteoporasis",
        "ChronicCond_rheumatoidarthritis", "ChronicCond_stroke"
    ]],
    StructField("IPAnnualReimbursementAmt", DoubleType(), True),
    StructField("IPAnnualDeductibleAmt", DoubleType(), True),
    StructField("OPAnnualReimbursementAmt", DoubleType(), True),
    StructField("OPAnnualDeductibleAmt", DoubleType(), True)
])

source_tables = {
    "inpatient": "health.default.train_inpatientdata_1542865627584",
    "outpatient": "health.default.train_outpatientdata_1542865627584",
    "beneficiary": "health.default.train_beneficiarydata_1542865627584",
    "fraud": "health.default.train_1542865627584"
}

bronze_tables = {
    "inpatient": "health.default.bronze_claims_inpatient",
    "outpatient": "health.default.bronze_claims_outpatient",
    "beneficiary": "health.default.bronze_beneficiaries",
    "fraud": "health.default.bronze_fraud_labels"
}


def cast_with_null_tolerance(column_name, target_type):
    cleaned = F.when(
        F.trim(F.upper(F.col(column_name).cast("string"))).isin("", "NA", "NULL", "NAN"),
        F.lit(None)
    ).otherwise(F.col(column_name))
    return cleaned.cast(target_type).alias(column_name)


def select_to_schema(df, schema):
    return df.select(*[cast_with_null_tolerance(field.name, field.dataType) for field in schema.fields])


# Load inpatient claims from table
df_inpatient = select_to_schema(spark.table(source_tables["inpatient"]), inpatient_schema) \
    .withColumn("claim_type", F.lit("inpatient")) \
    .withColumn("ingestion_timestamp", F.current_timestamp())

df_inpatient.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("claim_type") \
    .saveAsTable(bronze_tables["inpatient"])

# Load outpatient claims from table
df_outpatient = select_to_schema(spark.table(source_tables["outpatient"]), outpatient_schema) \
    .withColumn("claim_type", F.lit("outpatient")) \
    .withColumn("ingestion_timestamp", F.current_timestamp())

df_outpatient.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("claim_type") \
    .saveAsTable(bronze_tables["outpatient"])

# Load beneficiary data from table
df_beneficiary = select_to_schema(spark.table(source_tables["beneficiary"]), beneficiary_schema) \
    .withColumn("ingestion_timestamp", F.current_timestamp())

df_beneficiary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(bronze_tables["beneficiary"])

# Load fraud labels from table
df_fraud_labels = spark.table(source_tables["fraud"]).select(
    F.col("Provider").cast(StringType()).alias("Provider"),
    F.when(F.trim(F.upper(F.col("PotentialFraud"))) == "YES", F.lit(1)).otherwise(F.lit(0)).alias("is_fraud")
)

df_fraud_labels.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(bronze_tables["fraud"])

print("Bronze tables created:")
for table_name in bronze_tables.values():
    print(f"  {table_name}")

In [0]:
# Check for duplicates
def check_duplicates(table_name, key_columns):
    df = spark.table(table_name)
    total = df.count()
    distinct = df.select(*key_columns).distinct().count()

    print(f"{table_name}:")
    print(f"  Total records: {total}")
    print(f"  Distinct keys: {distinct}")
    print(f"  Duplicates: {total - distinct}")

check_duplicates("health.default.bronze_claims_inpatient", ["ClaimID"])
check_duplicates("health.default.bronze_claims_outpatient", ["ClaimID"])
check_duplicates("health.default.bronze_beneficiaries", ["BeneID"])

# Check date validity
df_inpatient = spark.table("health.default.bronze_claims_inpatient")
invalid_dates = df_inpatient.filter(
    (F.col("ClaimEndDt") < F.col("ClaimStartDt")) |
    (F.col("DischargeDt") < F.col("AdmissionDt"))
).count()

print(f"Invalid date ranges: {invalid_dates}")

# Missing value analysis
from pyspark.sql.functions import col, count, when

def missing_value_report(table_name):
    df = spark.table(table_name)
    total_rows = df.count()

    missing_stats = df.select([
        (count(when(col(c).isNull(), c)) / total_rows * 100).alias(c)
        for c in df.columns
    ])

    missing_pct = missing_stats.first().asDict()
    significant_missing = {k: v for k, v in missing_pct.items() if v > 5}

    print(f"\n{table_name} - Columns with >5% missing:")
    for col_name, pct in sorted(significant_missing.items(), key=lambda x: x[1], reverse=True):
        print(f"  {col_name}: {pct:.2f}%")

missing_value_report("health.default.bronze_claims_inpatient")
missing_value_report("health.default.bronze_beneficiaries")